# Plant Dataset — Decoy Strategy Comparison
Comparison of 4 decoy strategies (score_coord, score_coord_noise, nearest_neighbor, nearest_neighbor_noise) for accuracy estimation via Mix-Max FDR on the plant-daniella dataset.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import os
import tarfile
import io

print(f"NumPy {np.__version__}  PyTorch {torch.__version__}")


## Configuration

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

plt.rcParams.update({
    'font.size': 13, 'axes.titlesize': 14, 'axes.labelsize': 13,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 9,
    'figure.titlesize': 15, 'font.family': 'serif',
    'figure.dpi': 100, 'savefig.dpi': 150, 'savefig.bbox': 'tight',
})

os.makedirs('figures', exist_ok=True)


## Utility functions (calibration, baselines)

In [ ]:
def negentropy(logits):
    if isinstance(logits, np.ndarray):
        logits = torch.from_numpy(logits).float()
    probs = torch.softmax(logits, dim=1)
    entropy = -(probs * torch.log(probs + 1e-10)).sum(dim=1)
    return np.log(logits.shape[1]) - entropy

def calibration_temp(logits, labels, num_bins=15):
    if isinstance(logits, np.ndarray): logits = torch.from_numpy(logits).float()
    if isinstance(labels, np.ndarray): labels = torch.from_numpy(labels).long()
    temps = torch.linspace(0.1, 5.0, 50)
    best_temp, best_ece = 1.0, float('inf')
    for temp in temps:
        p = torch.softmax(logits / temp, dim=1)
        conf, pred = p.max(1); acc = (pred == labels).float()
        bins = torch.linspace(0, 1, num_bins + 1)
        ece = sum(
            ((conf > bins[i]) & (conf <= bins[i+1])).float().mean() *
            abs(conf[(conf > bins[i]) & (conf <= bins[i+1])].mean() -
                acc[(conf > bins[i]) & (conf <= bins[i+1])].mean()).item()
            for i in range(num_bins)
            if ((conf > bins[i]) & (conf <= bins[i+1])).sum() > 0
        )
        if ece < best_ece: best_ece, best_temp = ece, temp.item()
    return best_temp

def _to_tensor(x):
    return torch.from_numpy(x).float() if isinstance(x, np.ndarray) else x

def predict_ATC_maxconf(src_logits, src_labels, tgt_logits):
    src_logits = _to_tensor(src_logits); src_labels = _to_tensor(src_labels).long()
    tgt_logits = _to_tensor(tgt_logits)
    src_sc = torch.softmax(src_logits, 1).amax(1)
    tgt_sc = torch.softmax(tgt_logits, 1).amax(1)
    thr = torch.sort(src_sc).values[-(src_logits.argmax(1) == src_labels).sum()]
    return (tgt_sc > thr).float().mean().item()

def predict_ATC_negent(src_logits, src_labels, tgt_logits):
    src_logits = _to_tensor(src_logits); src_labels = _to_tensor(src_labels).long()
    tgt_logits = _to_tensor(tgt_logits)
    src_sc = negentropy(src_logits); tgt_sc = negentropy(tgt_logits)
    thr = torch.sort(src_sc).values[-(src_logits.argmax(1) == src_labels).sum()]
    return (tgt_sc > thr).float().mean().item()

def predict_AC(src_logits, src_labels, tgt_logits):
    return torch.softmax(_to_tensor(tgt_logits), 1).amax(1).mean().item()

def predict_DOC(src_logits, src_labels, tgt_logits):
    sl = _to_tensor(src_logits); sb = _to_tensor(src_labels).long(); tl = _to_tensor(tgt_logits)
    src_conf = torch.softmax(sl, 1).amax(1).mean().item()
    tgt_conf = torch.softmax(tl, 1).amax(1).mean().item()
    src_acc  = (sl.argmax(1) == sb).float().mean().item()
    return src_acc + (tgt_conf - src_conf)

BASELINE_METHODS = {'ATC': predict_ATC_maxconf, 'ATC-NE': predict_ATC_negent,
                    'AC': predict_AC, 'DOC': predict_DOC}

try:
    import ot
    def predict_COT(sl, sb, tl):
        sl = _to_tensor(sl); sb = _to_tensor(sb).long(); tl = _to_tensor(tl)
        nc = sl.shape[1]
        lbl_dist = F.one_hot(sb, nc).float().mean(0)
        tp = torch.softmax(tl, 1)
        cost = torch.stack([(tp - F.one_hot(torch.tensor(k), nc).float()).abs().sum(1) / 2 for k in range(nc)], 1)
        ot_plan = ot.emd(np.ones(len(tp)) / len(tp), lbl_dist.numpy(), cost.numpy())
        ot_cost = (ot_plan * cost.numpy()).sum()
        return 1 - (ot_cost + torch.softmax(sl, 1).amax(1).mean().item() - (sl.argmax(1) == sb).float().mean().item())
    BASELINE_METHODS['COT'] = predict_COT
    print("COT available")
except ImportError:
    print("COT unavailable (pip install POT)")
print(f"Baseline methods: {list(BASELINE_METHODS.keys())}")


## Flow model — building blocks

In [ ]:
class RobustFeatureNormalizer(nn.Module):
    def __init__(self, feature_dim, clip_val=5.0, momentum=0.01, eps=1e-6):
        super().__init__()
        self.clip_val = clip_val; self.momentum = momentum; self.eps = eps
        self.register_buffer('running_median', torch.zeros(feature_dim))
        self.register_buffer('running_iqr',    torch.ones(feature_dim))
        self.register_buffer('initialized',    torch.tensor(False))

    @torch.no_grad()
    def _update_stats(self, x):
        bm = x.median(0).values
        bi = (torch.quantile(x, 0.75, 0) - torch.quantile(x, 0.25, 0)).clamp(min=self.eps)
        if not self.initialized:
            self.running_median.copy_(bm); self.running_iqr.copy_(bi)
            self.initialized.fill_(True)
        else:
            self.running_median.mul_(1 - self.momentum).add_(bm * self.momentum)
            self.running_iqr.mul_(1 - self.momentum).add_(bi * self.momentum)

    def forward(self, x):
        if self.training: self._update_stats(x)
        if not self.initialized: return torch.tanh(x * 0.01)
        xn = ((x - self.running_median) / (self.running_iqr + self.eps)).clamp(-self.clip_val, self.clip_val)
        return torch.tanh(xn / self.clip_val)


class ActNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.log_scale = nn.Parameter(torch.zeros(dim))
        self.bias      = nn.Parameter(torch.zeros(dim))
        self.register_buffer('initialized', torch.tensor(False))

    def forward(self, x, reverse=False):
        if not self.initialized and not reverse:
            with torch.no_grad():
                self.bias.data      = -x.mean(0)
                self.log_scale.data = -x.std(0).clamp(min=1e-6).log()
            self.initialized.fill_(True)
        if not reverse:
            return (x + self.bias) * self.log_scale.exp(), self.log_scale.sum().expand(x.size(0))
        return x * (-self.log_scale).exp() - self.bias, -self.log_scale.sum().expand(x.size(0))


class CouplingLayer(nn.Module):
    def __init__(self, dim, feature_dim, hidden_dim=256, mask_type='first_half'):
        super().__init__()
        self.mask_type = mask_type
        self.d_in  = dim // 2 if mask_type == 'first_half' else dim - dim // 2
        self.d_out = dim - dim // 2 if mask_type == 'first_half' else dim // 2
        self.net = nn.Sequential(
            nn.Linear(self.d_in + feature_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.GELU())
        self.scale_head     = nn.Sequential(nn.Linear(hidden_dim // 2, self.d_out), nn.Tanh())
        self.translate_head = nn.Linear(hidden_dim // 2, self.d_out)
        nn.init.zeros_(self.scale_head[0].weight); nn.init.zeros_(self.scale_head[0].bias)
        nn.init.zeros_(self.translate_head.weight); nn.init.zeros_(self.translate_head.bias)

    def _split(self, x):
        return (x[:, :self.d_in], x[:, self.d_in:]) if self.mask_type == 'first_half'                else (x[:, self.d_out:], x[:, :self.d_out])

    def _merge(self, x1, x2):
        return torch.cat([x1, x2], 1) if self.mask_type == 'first_half' else torch.cat([x2, x1], 1)

    def forward(self, x, features, reverse=False):
        x1, x2 = self._split(x); h = self.net(torch.cat([x1, features], 1))
        s, t = self.scale_head(h), self.translate_head(h)
        if not reverse:
            return self._merge(x1, x2 * torch.exp(s) + t), s.sum(1)
        return self._merge(x1, (x2 - t) * torch.exp(-s)), -s.sum(1)


## ScoreShiftFlow

In [ ]:
class ScoreShiftFlow(nn.Module):
    def __init__(self, score_dim=10, feature_dim=640, n_flows=12, hidden_dim=256,
                 encoder_dim=128, clip_val=5.0):
        super().__init__()
        self.score_dim = score_dim
        self._log_2pi  = float(np.log(2 * np.pi))
        self.feature_norm    = RobustFeatureNormalizer(feature_dim, clip_val=clip_val, momentum=0.01)
        self.feature_encoder = nn.Sequential(
            nn.Linear(feature_dim, 256), nn.LayerNorm(256), nn.GELU(),
            nn.Linear(256, encoder_dim), nn.LayerNorm(encoder_dim), nn.GELU())
        self.layers = nn.ModuleList()
        for i in range(n_flows):
            mask = 'first_half' if i % 2 == 0 else 'second_half'
            self.layers.append(CouplingLayer(score_dim, encoder_dim, hidden_dim, mask))
            if i < n_flows - 1:
                self.layers.append(ActNorm(score_dim))

    def encode(self, f): return self.feature_encoder(self.feature_norm(f))

    def forward(self, scores, features, reverse=False):
        enc = self.encode(features)
        ld  = torch.zeros(scores.size(0), device=scores.device)
        if not reverse:
            x = scores
            for layer in self.layers:
                x, d = layer(x, reverse=False) if isinstance(layer, ActNorm)                        else layer(x, enc, reverse=False)
                ld += d
            return x, ld
        z = scores
        for layer in reversed(self.layers):
            z, d = layer(z, reverse=True) if isinstance(layer, ActNorm)                    else layer(z, enc, reverse=True)
            ld += d
        return z, ld

    def log_prob(self, scores, features):
        z, ld = self.forward(scores, features)
        return -0.5 * (z ** 2).sum(1) - 0.5 * self.score_dim * self._log_2pi + ld

    def sample(self, features):
        z = torch.randn(features.size(0), self.score_dim, device=features.device)
        s, _ = self.forward(z, features, reverse=True)
        return s


## ScoreShiftFlowWrapper

In [ ]:
class ScoreShiftFlowWrapper(nn.Module):
    def __init__(self, num_classes=10, n_flows=12, feature_dim=640,
                 hidden_dim=256, encoder_dim=128, clip_val=5.0):
        super().__init__()
        self.num_classes = num_classes
        self.flow = ScoreShiftFlow(num_classes, feature_dim, n_flows, hidden_dim, encoder_dim, clip_val)

    def train_flow(self, score_dataset, epochs=30, lr=3e-4, batch_size=256,
                   device='cuda', patience=5, grad_clip=1.0):
        self.flow.to(device).train()
        loader    = DataLoader(score_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
        optimizer = torch.optim.AdamW(self.flow.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr * 0.01)
        best_loss, best_state, no_improve = float('inf'), None, 0
        for epoch in range(epochs):
            total = 0.0; n = 0
            for _, feats, decoys, _ in loader:
                feats = feats.to(device); decoys = decoys.to(device)
                loss = -self.flow.log_prob(decoys, feats).mean()
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(self.flow.parameters(), grad_clip)
                optimizer.step(); total += loss.item(); n += 1
            scheduler.step(); avg = total / max(n, 1)
            if (epoch + 1) % 5 == 0:
                print(f"  Epoch {epoch+1:3d}/{epochs}  loss={avg:.4f}  lr={scheduler.get_last_lr()[0]:.2e}")
            if avg < best_loss - 1e-4:
                best_loss = avg; no_improve = 0
                best_state = {k: v.clone() for k, v in self.flow.state_dict().items()}
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f"  Early stop at epoch {epoch+1}")
                    self.flow.load_state_dict(best_state); break
        if best_state: self.flow.load_state_dict(best_state)
        print(f"  Done. Best loss: {best_loss:.4f}")
        return self

    def generate_decoys(self, score_dataset, device='cuda'):
        self.flow.to(device).eval()
        cnn_l, dc_l, lb_l = [], [], []
        loader = DataLoader(score_dataset, batch_size=256, shuffle=False, num_workers=0)
        with torch.no_grad():
            for cnn_sc, feats, _, labels in loader:
                feats = feats.to(device)
                dc_l.append(self.flow.sample(feats).cpu().numpy())
                cnn_l.append(cnn_sc.numpy()); lb_l.append(labels.numpy())
        return np.concatenate(cnn_l), np.concatenate(dc_l), np.concatenate(lb_l)


## ScoreFeatureDataset

In [ ]:
class ScoreFeatureDataset(torch.utils.data.Dataset):
    def __init__(self, cnn_scores, features, target_decoy_scores, labels):
        self.cnn_scores          = cnn_scores
        self.features            = features
        self.target_decoy_scores = target_decoy_scores
        self.labels              = labels
    def __len__(self): return len(self.cnn_scores)
    def __getitem__(self, i):
        return self.cnn_scores[i], self.features[i], self.target_decoy_scores[i], self.labels[i]


## Pool building functions

In [ ]:
MIN_POOL = 30

def build_error_conditioned_pools(train_scores, train_labels, num_classes,
                                   verbose=True, max_pool_size=10_000):
    """pool_score[c] = score_c on examples where argmax=c AND label≠c (fallback: label≠c)."""
    rng = np.random.default_rng(0)
    pred_classes = train_scores.argmax(axis=1)
    pool_score = {}
    if verbose:
        print(f"Building pools  (train acc={(pred_classes == train_labels).mean():.4f})")
    for c in range(num_classes):
        err_mask = (pred_classes == c) & (train_labels != c)
        cands = train_scores[err_mask, c] if err_mask.sum() >= MIN_POOL                 else train_scores[train_labels != c, c]
        pool_score[c] = cands[rng.choice(len(cands), size=max_pool_size, replace=False)]                         if len(cands) > max_pool_size else cands
        if verbose:
            print(f"  class {c}: n={len(pool_score[c]):6d}  [{pool_score[c].min():.3f},{pool_score[c].max():.3f}]")
    return pool_score


def _build_decoy_score_coord(sc_np, pool_score, rng):
    """Replace coord c_hat by a random draw from pool_score[c_hat]."""
    pred = sc_np.argmax(axis=1); dc = sc_np.copy()
    for c in range(sc_np.shape[1]):
        mask = pred == c
        if mask.any() and len(pool_score.get(c, [])) > 0:
            dc[mask, c] = rng.choice(pool_score[c], size=mask.sum(), replace=True)
    return dc


## Data loading utilities

In [ ]:
def load_precomputed_data(path):
    """Load (features, logits, labels) from .pt or .tar.xz archive."""
    if path.endswith(('.tar.xz', '.tar.gz', '.tar.bz2', '.tar')):
        with tarfile.open(path, 'r:*') as tar:
            members = [m for m in tar.getmembers() if m.name.endswith('.pt')]
            if not members: raise FileNotFoundError(f"No .pt inside {path}")
            data = torch.load(io.BytesIO(tar.extractfile(members[0]).read()),
                              map_location='cpu', weights_only=False)
    else:
        data = torch.load(path, map_location='cpu', weights_only=False)
    print(list(data.keys()))
    feat_key  = 'hidden_features' if 'hidden_features' in data else 'test_hidden_features'
    logit_key = 'logits'          if 'logits'          in data else 'test_logits'
    label_key = 'labels'          if 'labels'          in data else 'test_labels'
    return data[feat_key].float(), data[logit_key].float(), data[label_key].long()


## Load precomputed plant dataset

In [ ]:
TRAIN_DATA_PATH = '/kaggle/input/datasets/arinaromashkina/plantlet/train_data2 (1).tar.xz'
TEST_DATA_PATH  = '/kaggle/input/datasets/arinaromashkina/plantlet/test_data2.tar.xz'

train_features, train_logits, train_labels_t = load_precomputed_data(TRAIN_DATA_PATH)
test_features,  test_logits,  test_labels_t  = load_precomputed_data(TEST_DATA_PATH)

train_scores_raw = train_logits.numpy()
train_labels_raw = train_labels_t.numpy()

NUM_CLASSES = train_logits.shape[1]
FEATURE_DIM = train_features.shape[1]

print(f"Train: logits={train_logits.shape}  features={train_features.shape}")
print(f"Test:  logits={test_logits.shape}   features={test_features.shape}")
print(f"NUM_CLASSES={NUM_CLASSES}  FEATURE_DIM={FEATURE_DIM}")
print(f"Train acc={(train_scores_raw.argmax(1) == train_labels_raw).mean():.4f}")


### Dataset statistics

In [ ]:
# ── True label distribution (train + test) ───────────────────────────────────
for split, labels in [('TRAIN', train_labels_raw), ('TEST', lb_np)]:
    counts = np.bincount(labels)
    nonzero = counts[counts > 0]
    top_cls = np.argsort(counts)[::-1][:10]
    print(f"\n{split}: {len(labels)} samples, {(counts > 0).sum()} classes with data")
    print(f"  samples/class — min: {nonzero.min()}  median: {int(np.median(nonzero))}  "
          f"max: {nonzero.max()}  mean: {nonzero.mean():.1f}")
    print(f"  Top 10 classes by true label count:")
    for c in top_cls:
        if counts[c] == 0: break
        print(f"    class {c:>4d}: {counts[c]:>5d}  ({counts[c]/len(labels)*100:.1f}%)")

## Build error-conditioned decoy pools

In [ ]:
pool_score = build_error_conditioned_pools(
    train_scores_raw, train_labels_raw, NUM_CLASSES, verbose=True)


## Strategy comparison — configuration & helper functions

In [ ]:
NOISE_STD = 0.5

STRATEGIES = [
    ('score_coord',            0.0),
    ('score_coord_noise',      NOISE_STD),
    ('nearest_neighbor',       0.0),
    ('nearest_neighbor_noise', NOISE_STD),
]
STRATEGY_LABELS = {
    'score_coord':            'Random (SC)',
    'score_coord_noise':      'Random + noise',
    'nearest_neighbor':       'Nearest-neighbor',
    'nearest_neighbor_noise': 'NN + noise',
}
STRATEGY_COLORS = {
    'score_coord':            '#1976D2',
    'score_coord_noise':      '#42A5F5',
    'nearest_neighbor':       '#E65100',
    'nearest_neighbor_noise': '#FF8A65',
}

MAX_NN_POOL = 5000

def build_error_vector_pool(train_scores, train_labels, num_classes,
                            min_pool=30, max_pool_size=MAX_NN_POOL):
    """Pool of full logit vectors for NN strategy, capped at max_pool_size per class."""
    rng  = np.random.default_rng(0)
    pred = train_scores.argmax(axis=1)
    pool = {}
    for k in range(num_classes):
        err_mask = (pred == k) & (train_labels != k)
        vecs = train_scores[err_mask] if err_mask.sum() >= min_pool \
               else train_scores[train_labels != k]
        if len(vecs) > max_pool_size:
            vecs = vecs[rng.choice(len(vecs), size=max_pool_size, replace=False)]
        pool[k] = vecs
        print(f"  NN pool class {k}: {len(vecs)} vectors")
    return pool


NN_BATCH = 1024

def _build_decoy_nearest_neighbor(sc_np, pool_error_vectors, rng, noise_std=0.0):
    """NN decoy with batched distance computation to avoid OOM."""
    n, C = sc_np.shape; pred = sc_np.argmax(1); dc = sc_np.copy(); coords = np.arange(C)
    for k in range(C):
        mask = pred == k
        if not mask.any(): continue
        pk = pool_error_vectors.get(k)
        if pk is None or len(pk) == 0: continue
        comp = coords[coords != k]
        S = sc_np[mask][:, comp]
        V = pk[:, comp]
        V_sq = (V ** 2).sum(1)
        best_idx = np.empty(S.shape[0], dtype=np.intp)
        for start in range(0, S.shape[0], NN_BATCH):
            end = min(start + NN_BATCH, S.shape[0])
            Sb = S[start:end]
            dists = (Sb ** 2).sum(1, keepdims=True) + V_sq[None, :] - 2 * Sb @ V.T
            best_idx[start:end] = np.maximum(dists, 0).argmin(1)
        dc[mask, k] = pk[best_idx, k]
    if noise_std > 0: dc += rng.normal(0, noise_std, dc.shape)
    return dc


def apply_strategy(scores_np, pool_score, pool_error_vectors, strategy, noise_std):
    rng = np.random.default_rng(42)
    _n  = noise_std if strategy in ('score_coord_noise', 'nearest_neighbor_noise') else 0.0
    if strategy in ('nearest_neighbor', 'nearest_neighbor_noise'):
        return _build_decoy_nearest_neighbor(scores_np, pool_error_vectors, rng, _n)
    dc = _build_decoy_score_coord(scores_np, pool_score, rng)
    if _n > 0: dc += rng.normal(0, _n, dc.shape)
    return dc


def compute_fdr_acc_curves(scores_np, decoy_np, labels_np, pi0=0.0):
    n = len(labels_np)
    pred_sc = scores_np.max(1); pred_lb = scores_np.argmax(1); dc_sc = decoy_np.max(1)
    correct = (pred_lb == labels_np).astype(int)
    sidx = np.argsort(pred_sc); ps = pred_sc[sidx]; cs = correct[sidx]

    # True FDR
    FD = 1 - cs; FC = np.cumsum(FD[::-1])[::-1]; DC = np.arange(n, 0, -1)
    QVAL_true = np.clip(np.minimum.accumulate(np.clip(FC / DC, 0, 1)), 0, 1)

    # TDC
    tsc = np.maximum(pred_sc, dc_sc); twin = (pred_sc > dc_sc).astype(int)
    ti = np.argsort(tsc); FC_t = np.cumsum((1 - twin[ti])[::-1])[::-1]
    DC_t = np.maximum(DC - FC_t, 1)
    QVAL_TDC = np.clip(np.minimum.accumulate(np.clip(FC_t / DC_t, 0, 1)), 0, 1)

    # Mix-Max
    sd = np.sort(dc_sc); uz, cz = np.unique(dc_sc, return_counts=True)
    nuz = len(uz)
    PW = np.clip((np.searchsorted(ps, uz, 'left') - pi0 * np.searchsorted(sd, uz, 'left')) / ((1-pi0)*n), 0, 1)
    PY = np.clip(np.searchsorted(sd, uz, 'left') / n, 0, 1)
    Rj = np.clip(np.divide(PW, PY, out=np.zeros_like(PW), where=PY > 0), 0, 1)
    fdr = np.zeros(n)
    for i, T in enumerate(ps[::-1]):
        D = i + 1
        F0 = pi0 * (dc_sc > T).sum()
        zi = np.searchsorted(uz, T, 'left')
        F1 = 0.0 if zi >= nuz else (1-pi0) * (Rj[zi:] * cz[zi:]).sum()
        fdr[i] = (F0 + F1) / D if D > 0 else 0
    QVAL_mm = np.clip(np.minimum.accumulate(np.clip(fdr, 0, 1)[::-1]), 0, 1)

    # Acc curves
    pi0_tdc = float(np.clip(QVAL_TDC[0], 0, 1))
    pi0_mm  = float(np.clip(QVAL_mm[0],  0, 1))
    At = np.zeros(n); Ae = np.zeros(n); Am = np.zeros(n)
    for i in range(n):
        At[i] = (cs[i:].sum() + (1 - cs[:i]).sum()) / n
        acc = n - i
        Ae[i] = np.clip((acc*(1 - QVAL_TDC[i]) + n*pi0_tdc - acc*QVAL_TDC[i]) / n, 0, 1)
        Am[i] = np.clip((acc*(1 - QVAL_mm[i])  + n*pi0_mm  - acc*QVAL_mm[i])  / n, 0, 1)
    At = np.clip(At, 0, 1)
    r  = np.arange(n) / n
    tp = int(cs.sum()); TPi = np.cumsum(cs[::-1])[::-1]; Di = DC
    acc_st = float(At[0]); acc_ta = float(At.max())
    acc_st_mm = float(Am[0]); acc_ta_mm = float(Am.max())
    return dict(
        normalized_rank=r, pred_scores_sorted=ps, pred_scores=pred_sc, decoy_scores=dc_sc,
        QVAL_true=QVAL_true, QVAL_TDC=QVAL_TDC, QVAL_mixmax=QVAL_mm,
        Acc_true=At, Acc_est=Ae, Acc_est_MM=Am,
        precision_true=np.where(Di>0, TPi/Di, 0), recall_true=TPi/max(tp,1),
        precision_est=np.clip(1-QVAL_mm, 0, 1),
        recall_est=np.clip((1-QVAL_mm)*Di/max(tp,1), 0, 1),
        correct=correct, pred_label=pred_lb, labels=labels_np,
        true_acc=float(correct.mean()), acc_st_true=acc_st, acc_ta_true=acc_ta,
        acc_st_est_mm=acc_st_mm, acc_ta_est_mm=acc_ta_mm,
        err_st_mm=abs(acc_st_mm - acc_st), err_ta_mm=abs(acc_ta_mm - acc_ta), n=n,
    )

## Build NN pool and prepare test arrays

In [ ]:
print("Building NN error vector pool...")
pool_error_vectors = build_error_vector_pool(train_scores_raw, train_labels_raw, NUM_CLASSES)

sc_np = test_logits.numpy()
ft_np = test_features.numpy()
lb_np = test_labels_t.numpy()

print(f"Test: n={len(lb_np)}  acc={(sc_np.argmax(1)==lb_np).mean():.4f}")
n_strats = len(STRATEGIES)


## Run raw decoy strategies

In [ ]:
raw_results = {}
for strat_name, noise_std in STRATEGIES:
    dc   = apply_strategy(sc_np, pool_score, pool_error_vectors, strat_name, noise_std)
    crv  = compute_fdr_acc_curves(sc_np, dc, lb_np)
    raw_results[strat_name] = crv
    print(f"  {strat_name:<25}  true_acc={crv['true_acc']:.3f}  "
          f"err_st={crv['err_st_mm']:.3f}  err_ta={crv['err_ta_mm']:.3f}")


### Class-wise FDR diagnostic

Evaluate FDR/Acc **per predicted class** for the top 3 most abundant classes.  
- If per-class curves are good but aggregate is bad → problem is cross-class competition (many classes dilute decoy quality).  
- If per-class curves are already bad → pool itself doesn't match test distribution for that class.

In [ ]:
# Find top 3 most-predicted classes
pred_test = sc_np.argmax(1)
class_counts = np.bincount(pred_test, minlength=NUM_CLASSES)
top3 = np.argsort(class_counts)[::-1][:3]
print("Top 3 predicted classes:")
for c in top3:
    n_c = class_counts[c]
    acc_c = (pred_test[pred_test == c] == lb_np[pred_test == c]).mean()
    print(f"  class {c}: n={n_c}  ({n_c/len(pred_test)*100:.1f}%)  acc={acc_c:.3f}  pool_score n={len(pool_score[c])}")

# Compute per-class FDR for score_coord strategy (representative)
strat_for_classwise = 'score_coord'
dc_full = apply_strategy(sc_np, pool_score, pool_error_vectors, strat_for_classwise, 0.0)

classwise = {}
for c in top3:
    mask = pred_test == c
    if mask.sum() < 20:
        print(f"  class {c}: too few samples ({mask.sum()}), skipping")
        continue
    classwise[c] = compute_fdr_acc_curves(sc_np[mask], dc_full[mask], lb_np[mask])

# Also compute aggregate for comparison
agg = raw_results[strat_for_classwise]

# ── Plot: per-class FDR + Acc vs aggregate ───────────────────────────────────
n_cls = len(classwise)
fig, axes = plt.subplots(2, n_cls + 1, figsize=(5 * (n_cls + 1), 8))

colors_cls = ['#E53935', '#1E88E5', '#43A047']

# Aggregate column
ax = axes[0][0]
r = agg['normalized_rank']
ax.plot(r, agg['QVAL_true'],   'k--', lw=2,   label='True FDR')
ax.plot(r, agg['QVAL_mixmax'], color='#1976D2', lw=1.8, label=f'MixMax err={agg["err_st_mm"]:.3f}')
ax.set_title(f'ALL CLASSES (n={agg["n"]})\nacc={agg["true_acc"]:.3f}')
ax.set_ylabel('q-value'); ax.set_xlabel('Fraction accepted')
ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.3); ax.set_ylim(0, 1.05)

ax = axes[1][0]
ax.plot(r, agg['Acc_true'],   'k--', lw=2,   label='True Acc')
ax.plot(r, agg['Acc_est_MM'], color='#1976D2', lw=1.8, label=f'MixMax err={agg["err_ta_mm"]:.3f}')
ax.set_ylabel('Accuracy'); ax.set_xlabel('Fraction accepted')
ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.3)

# Per-class columns
for col_i, (c, color) in enumerate(zip(classwise.keys(), colors_cls)):
    cv = classwise[c]
    r_c = cv['normalized_rank']
    n_c = cv['n']; acc_c = cv['true_acc']

    ax = axes[0][col_i + 1]
    ax.plot(r_c, cv['QVAL_true'],   'k--', lw=2,  label='True FDR')
    ax.plot(r_c, cv['QVAL_mixmax'], color=color, lw=1.8, label=f'MixMax err={cv["err_st_mm"]:.3f}')
    ax.set_title(f'CLASS {c} (n={n_c})\nacc={acc_c:.3f}  pool={len(pool_score[c])}')
    ax.set_xlabel('Fraction accepted')
    ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.3); ax.set_ylim(0, 1.05)

    ax = axes[1][col_i + 1]
    ax.plot(r_c, cv['Acc_true'],   'k--', lw=2,  label='True Acc')
    ax.plot(r_c, cv['Acc_est_MM'], color=color, lw=1.8, label=f'MixMax err={cv["err_ta_mm"]:.3f}')
    ax.set_xlabel('Fraction accepted')
    ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.3)

plt.suptitle(f'Plant — Class-wise FDR/Acc  (strategy: {strat_for_classwise})', fontsize=14)
plt.tight_layout(); plt.show()

# Summary table
print(f"\n{'Class':<12} {'n':>6}  {'acc':>6}  {'pool_n':>7}  {'MAE_ST':>8}  {'MAE_TA':>8}")
print('-' * 55)
print(f"{'ALL':<12} {agg['n']:>6}  {agg['true_acc']:>6.3f}  {'':>7}  {agg['err_st_mm']:>8.4f}  {agg['err_ta_mm']:>8.4f}")
for c in classwise:
    cv = classwise[c]
    print(f"{'class '+str(c):<12} {cv['n']:>6}  {cv['true_acc']:>6.3f}  {len(pool_score[c]):>7}  {cv['err_st_mm']:>8.4f}  {cv['err_ta_mm']:>8.4f}")

### Per-class scatter: model score vs decoy score at coord c

For each of the top 3 classes, scatter `score[i, c]` (model logit at the predicted class) vs `decoy[i, c]` (the replaced coord).  
This shows directly whether the pool draws land around the diagonal for incorrect predictions — the core exchangeability requirement.

In [ ]:
# Per-class scatter: model score_c vs decoy score_c (at the predicted class coord)
dc_sc = dc_full  # score_coord decoys already computed above

fig, axes = plt.subplots(1, len(top3), figsize=(6 * len(top3), 5))
if len(top3) == 1: axes = [axes]

for ax, c in zip(axes, top3):
    mask = pred_test == c
    correct_c = (pred_test[mask] == lb_np[mask])

    model_at_c = sc_np[mask, c]
    decoy_at_c = dc_sc[mask, c]

    # Pool distribution for reference
    pool_c = pool_score[c]

    ax.scatter(model_at_c[correct_c],  decoy_at_c[correct_c],
               s=8, alpha=0.3, color='steelblue', label=f'correct ({correct_c.sum()})', rasterized=True)
    ax.scatter(model_at_c[~correct_c], decoy_at_c[~correct_c],
               s=12, alpha=0.6, color='crimson', label=f'incorrect ({(~correct_c).sum()})', rasterized=True)

    lo = min(model_at_c.min(), decoy_at_c.min(), pool_c.min()) - 0.3
    hi = max(model_at_c.max(), decoy_at_c.max(), pool_c.max()) + 0.3
    ax.plot([lo, hi], [lo, hi], 'k--', lw=0.8, alpha=0.5)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)

    # Show pool range as shaded band on y-axis
    p5, p95 = np.percentile(pool_c, [5, 95])
    ax.axhspan(p5, p95, alpha=0.12, color='orange', label=f'pool 5-95% [{p5:.1f}, {p95:.1f}]')
    ax.axhline(np.median(pool_c), color='orange', ls=':', lw=1.5, alpha=0.7, label=f'pool median={np.median(pool_c):.1f}')

    # Annotate median shift
    med_model_inc = np.median(model_at_c[~correct_c]) if (~correct_c).sum() > 0 else np.nan
    med_pool = np.median(pool_c)
    ax.set_title(f'Class {c}  (n={mask.sum()}, acc={correct_c.mean():.2f})\n'
                 f'pool_med={med_pool:.2f}  incorr_model_med={med_model_inc:.2f}  '
                 f'shift={med_pool - med_model_inc:+.2f}')
    ax.set_xlabel(f'Model score at class {c}')
    ax.set_ylabel(f'Decoy score at class {c}')
    ax.legend(fontsize=8, loc='upper left')
    ax.set_aspect('equal', 'box')
    ax.grid(ls='--', alpha=0.3)

plt.suptitle('Plant — Per-class scatter: model vs decoy at coord c  (score_coord strategy)', fontsize=13)
plt.tight_layout(); plt.show()

### Figure 1 — Score distributions

In [ ]:
ref = raw_results['score_coord']
bins = np.linspace(ref['pred_scores'].min() - 0.3, ref['pred_scores'].max() + 0.3, 60)
inc_mask = ref['correct'] == 0

fig, axes = plt.subplots(1, n_strats, figsize=(5*n_strats, 4))
for ax, (strat_name, _) in zip(axes, STRATEGIES):
    c = raw_results[strat_name]; color = STRATEGY_COLORS[strat_name]
    sns.histplot(c['pred_scores'],            bins=bins, stat='density', color='steelblue',
                 kde=True, fill=True, alpha=0.3, label='model',     ax=ax)
    sns.histplot(c['decoy_scores'],           bins=bins, stat='density', color=color,
                 kde=True, fill=True, alpha=0.4, label='decoy',     ax=ax)
    if inc_mask.any():
        sns.histplot(c['pred_scores'][inc_mask], bins=bins, stat='density', color='crimson',
                     kde=True, fill=True, alpha=0.3, label='incorrect', ax=ax)
    ax.set_title(f'{STRATEGY_LABELS[strat_name]}\ntrue_acc={c["true_acc"]:.3f}')
    ax.set_xlabel('Max logit'); ax.legend(fontsize=8)
    if ax is axes[0]: ax.set_ylabel('Density')
plt.suptitle('Plant — Score Distributions (Raw Decoy)', fontsize=13)
plt.tight_layout(); plt.show()


### Figure 2 — FDR curves

In [ ]:
fig, axes = plt.subplots(1, n_strats, figsize=(5*n_strats, 4))
for ax, (strat_name, _) in zip(axes, STRATEGIES):
    c = raw_results[strat_name]; r = c['normalized_rank']
    ax.plot(r, c['QVAL_true'],   color='gray',                     lw=1.5, ls='--', label='True FDR')
    ax.plot(r, c['QVAL_mixmax'], color=STRATEGY_COLORS[strat_name], lw=1.8,         label='Mix-Max')
    ax.plot(r, c['QVAL_TDC'],   color='navy',                     lw=1.2, ls=':',  label='TDC')
    ax.axhline(0.1, color='black', lw=0.8, ls=':', alpha=0.4)
    ax.set_title(f'{STRATEGY_LABELS[strat_name]}\nMAE_ST={c["err_st_mm"]:.3f}')
    ax.set_xlabel('Fraction accepted'); ax.legend(fontsize=8)
    ax.grid(ls='--', alpha=0.3); ax.set_ylim(0, 1.05)
    if ax is axes[0]: ax.set_ylabel('q-value')
plt.suptitle('Plant — FDR Curves', fontsize=13)
plt.tight_layout(); plt.show()


### Figure 3 — Accuracy curves overlay

In [ ]:
r = ref['normalized_rank']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(r, ref['Acc_true'], color='black', lw=2, ls='--', label='True Acc')
for strat_name, _ in STRATEGIES:
    c = raw_results[strat_name]
    ax.plot(r, c['Acc_est_MM'], color=STRATEGY_COLORS[strat_name], lw=1.8,
            label=f'{STRATEGY_LABELS[strat_name]} (MAE={c["err_st_mm"]:.3f})')
ax.set_xlabel('Fraction accepted'); ax.set_ylabel('Accuracy')
ax.set_title('Accuracy curves (MixMax)'); ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.4)

ax = axes[1]
ax.plot(r, ref['QVAL_true'], color='black', lw=2, ls='--', label='True FDR')
for strat_name, _ in STRATEGIES:
    c = raw_results[strat_name]
    ax.plot(r, c['QVAL_mixmax'], color=STRATEGY_COLORS[strat_name], lw=1.8, label=STRATEGY_LABELS[strat_name])
ax.set_xlabel('Fraction accepted'); ax.set_ylabel('q-value')
ax.set_title('FDR curves comparison'); ax.legend(fontsize=8)
ax.grid(ls='--', alpha=0.4); ax.set_ylim(0, 1.05)

plt.suptitle('Plant — Strategy Comparison', fontsize=13)
plt.tight_layout(); plt.show()


### Figure 4 — Scatter: decoy vs model score (TEST)

In [ ]:
corr_test = raw_results['score_coord']['correct'].astype(bool)
fig, axes = plt.subplots(1, n_strats, figsize=(5*n_strats, 4))
for ax, (strat_name, _) in zip(axes, STRATEGIES):
    c = raw_results[strat_name]; ms = c['pred_scores']; ds = c['decoy_scores']
    idx = np.random.default_rng(0).choice(len(ms), min(3000, len(ms)), replace=False)
    cor = corr_test[idx]
    ax.scatter(ms[idx][cor],  ds[idx][cor],  s=6, alpha=0.3, color='steelblue', label='correct',  rasterized=True)
    ax.scatter(ms[idx][~cor], ds[idx][~cor], s=6, alpha=0.5, color='crimson',   label='incorrect', rasterized=True)
    lims = [min(ms.min(), ds.min()) - 0.1, max(ms.max(), ds.max()) + 0.1]
    ax.plot(lims, lims, 'k--', lw=0.8, alpha=0.5); ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_title(STRATEGY_LABELS[strat_name]); ax.set_xlabel('Model score')
    ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.3)
    if ax is axes[0]: ax.set_ylabel('Decoy score')
plt.suptitle('Plant — Decoy vs Model Score (TEST)', fontsize=13)
plt.tight_layout(); plt.show()


### Figure 5 — Scatter: decoy vs model score (TRAINING)

Diagnoses distribution shift: if training looks OK (dots straddle the diagonal) but test is all below → pool doesn't cover test distribution.

In [ ]:
tr_sc = train_scores_raw; tr_lb = train_labels_raw
corr_tr = tr_sc.argmax(1) == tr_lb
fig, axes = plt.subplots(1, n_strats, figsize=(5*n_strats, 4))
for ax, (strat_name, noise_std) in zip(axes, STRATEGIES):
    dc = apply_strategy(tr_sc, pool_score, pool_error_vectors, strat_name, noise_std)
    ms = tr_sc.max(1); ds = dc.max(1)
    frac_above = (ds[~corr_tr] > ms[~corr_tr]).mean()
    idx = np.random.default_rng(0).choice(len(ms), min(3000, len(ms)), replace=False)
    cor = corr_tr[idx]
    ax.scatter(ms[idx][cor],  ds[idx][cor],  s=6, alpha=0.3, color='steelblue', label='correct',  rasterized=True)
    ax.scatter(ms[idx][~cor], ds[idx][~cor], s=6, alpha=0.6, color='crimson',   label='incorrect', rasterized=True)
    lims = [min(ms.min(), ds.min()) - 0.1, max(ms.max(), ds.max()) + 0.1]
    ax.plot(lims, lims, 'k--', lw=0.8, alpha=0.5); ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_title(f'{STRATEGY_LABELS[strat_name]}\n'
                 f'train_acc={corr_tr.mean():.3f}  decoy>model(incorr):{frac_above:.2f}')
    ax.set_xlabel('Model score'); ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.3)
    if ax is axes[0]: ax.set_ylabel('Decoy score')
plt.suptitle('Plant — TRAINING data scatter (Decoy vs Model Score)', fontsize=13)
plt.tight_layout(); plt.show()


### Figure 6 — Pool coverage diagnostic

Ideal: `pool ≥ incorrect` ≈ 0.5 (pool looks like the test error distribution). A shift means pool is too low/high relative to test.

In [ ]:
pred_cls_test = sc_np.argmax(1); corr_test_b = pred_cls_test == lb_np
s_chat = sc_np[np.arange(len(sc_np)), pred_cls_test]
tr_pred = tr_sc.argmax(1); tr_err = tr_pred != tr_lb
all_pool = np.concatenate([pool_score[c] for c in range(NUM_CLASSES)])
sc_tr_err = tr_sc[tr_err][np.arange(tr_err.sum()), tr_pred[tr_err]]

frac_pool = np.mean([np.mean(pool_score[c] >= s)
                     for s, c in zip(s_chat[~corr_test_b], pred_cls_test[~corr_test_b])
                     ]) if (~corr_test_b).sum() > 0 else float('nan')
shift = np.median(all_pool) - np.median(s_chat[~corr_test_b])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
ax.hist(sc_tr_err, bins=50, density=True, alpha=0.6, color='crimson',  label=f'train errors at c_hat (n={tr_err.sum()})')
ax.hist(all_pool,  bins=50, density=True, alpha=0.5, color='orange',   label=f'pool values (n={len(all_pool)})')
ax.set_title('TRAIN: error scores vs pool'); ax.set_xlabel('Score at c_hat'); ax.set_ylabel('Density')
ax.legend(fontsize=9); ax.grid(ls='--', alpha=0.3)

ax = axes[1]
ax.hist(s_chat[~corr_test_b], bins=50, density=True, alpha=0.6, color='crimson',   label='test incorrect')
ax.hist(s_chat[corr_test_b],  bins=50, density=True, alpha=0.4, color='steelblue', label='test correct')
ax.hist(all_pool,             bins=50, density=True, alpha=0.4, color='orange',    label='pool', linestyle='--')
ax.set_title(f'TEST: scores vs pool\npool ≥ incorrect: {frac_pool:.2f}  pool-median shift: {shift:+.2f}')
ax.set_xlabel('Score at c_hat'); ax.legend(fontsize=9); ax.grid(ls='--', alpha=0.3)

plt.suptitle('Plant — Pool coverage diagnostic', fontsize=13)
plt.tight_layout(); plt.show()
print(f'Pool coverage:  pool ≥ incorrect test: {frac_pool:.3f}  (ideal 0.5)')
print(f'  pool median: {np.median(all_pool):.3f}  incorrect test median: {np.median(s_chat[~corr_test_b]):.3f}  shift: {shift:+.3f}')


### Figure 7 — P-value diagnostics & pool calibration

Under H₀: `p_i = P(pool[c_hat] ≥ score_at_c_hat)` should be Uniform[0,1] for incorrect predictions.

In [ ]:
def compute_pvalues(sc_np, lb_np, pool_score):
    pred = sc_np.argmax(1); s = sc_np[np.arange(len(sc_np)), pred]
    pv = np.array([np.mean(pool_score[c] >= si) for si, c in zip(s, pred)])
    return pv, pred == lb_np, pred

def build_calibrated_pool(pool_score, test_scores_per_class, num_classes):
    """Shift+scale pool per class to match test median + IQR."""
    cal = {}
    for c in range(num_classes):
        pool = pool_score.get(c, np.array([])); tsc = test_scores_per_class.get(c, np.array([]))
        if len(pool) == 0 or len(tsc) == 0: cal[c] = pool; continue
        pm, tm = np.median(pool), np.median(tsc)
        pi = np.percentile(pool, 75) - np.percentile(pool, 25)
        ti = np.percentile(tsc,  75) - np.percentile(tsc,  25)
        cal[c] = pool + (tm - pm) if pi < 1e-6 or ti < 1e-6 else (pool - pm) * (ti / pi) + tm
    return cal

p_raw, corr_b, p_cls = compute_pvalues(sc_np, lb_np, pool_score)
tsc_per_cls = {c: sc_np[p_cls == c, c] for c in range(NUM_CLASSES) if (p_cls == c).sum() > 0}
pool_cal    = build_calibrated_pool(pool_score, tsc_per_cls, NUM_CLASSES)
p_cal, _, _ = compute_pvalues(sc_np, lb_np, pool_cal)

print(f'Raw pool:   incorrect mean_p={p_raw[~corr_b].mean():.3f}  correct mean_p={p_raw[corr_b].mean():.3f}')
print(f'Calibrated: incorrect mean_p={p_cal[~corr_b].mean():.3f}  correct mean_p={p_cal[corr_b].mean():.3f}  (ideal incorrect≈0.5)')

bins_p = np.linspace(0, 1, 21)
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

ax = axes[0]
ax.hist(p_raw[corr_b],  bins=bins_p, density=True, alpha=0.6, color='steelblue', label=f'Correct (n={corr_b.sum()})')
ax.hist(p_raw[~corr_b], bins=bins_p, density=True, alpha=0.6, color='crimson',   label=f'Incorrect (n={(~corr_b).sum()})')
ax.axhline(1, ls='--', color='black', lw=1.2, label='Uniform')
ax.set_title(f'Raw pool p-values\nincorrect mean={p_raw[~corr_b].mean():.3f}')
ax.set_xlabel('p-value'); ax.set_ylabel('Density'); ax.legend(fontsize=9); ax.grid(ls='--', alpha=0.3)

ax = axes[1]
ax.hist(p_cal[corr_b],  bins=bins_p, density=True, alpha=0.6, color='steelblue', label='Correct')
ax.hist(p_cal[~corr_b], bins=bins_p, density=True, alpha=0.6, color='crimson',   label='Incorrect')
ax.axhline(1, ls='--', color='black', lw=1.2, label='Uniform')
ax.set_title(f'Calibrated pool p-values\nincorrect mean={p_cal[~corr_b].mean():.3f}')
ax.set_xlabel('p-value'); ax.legend(fontsize=9); ax.grid(ls='--', alpha=0.3)

ax = axes[2]
for pv, lbl, col in [(p_raw[~corr_b], 'Raw (H0)', 'crimson'), (p_cal[~corr_b], 'Cal (H0)', 'darkorange')]:
    if len(pv): ax.plot(np.linspace(0,1,len(pv)), np.sort(pv), color=col, lw=1.8, label=lbl)
ax.plot([0,1],[0,1],'k--',lw=0.8,alpha=0.6,label='Uniform')
ax.set_xlabel('Uniform quantile'); ax.set_ylabel('Empirical p-value quantile')
ax.set_title('QQ plot — incorrect preds\n(H0 ideal = diagonal)'); ax.legend(fontsize=9); ax.grid(ls='--', alpha=0.3)

ax = axes[3]
cr = [p_raw[(p_cls==c)&~corr_b].mean() if ((p_cls==c)&~corr_b).sum()>0 else np.nan for c in range(NUM_CLASSES)]
cc = [p_cal[(p_cls==c)&~corr_b].mean() if ((p_cls==c)&~corr_b).sum()>0 else np.nan for c in range(NUM_CLASSES)]
x = np.arange(NUM_CLASSES)
ax.bar(x-0.2, cr, 0.4, color='crimson',   alpha=0.7, label='Raw pool')
ax.bar(x+0.2, cc, 0.4, color='darkorange', alpha=0.7, label='Calibrated')
ax.axhline(0.5, ls='--', color='black', lw=1, label='Ideal (0.5)')
ax.set_xlabel('Predicted class c_hat'); ax.set_ylabel('Mean p-value (incorrect only)')
ax.set_title('Per-class pool calibration'); ax.legend(fontsize=9); ax.grid(ls='--', alpha=0.3)

plt.suptitle('Plant — P-value diagnostics', fontsize=13)
plt.tight_layout(); plt.show()


### Effect of pool calibration on FDR / Accuracy

In [ ]:
dc_cal     = apply_strategy(sc_np, pool_cal, pool_error_vectors, 'score_coord', 0.0)
curves_cal = compute_fdr_acc_curves(sc_np, dc_cal, lb_np)
sc_ref = raw_results['score_coord']; r = sc_ref['normalized_rank']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
ax.plot(r, sc_ref['QVAL_true'],       'k--', lw=2,   label='True FDR')
ax.plot(r, sc_ref['QVAL_mixmax'],     color='#1976D2',   lw=1.8, label=f'Raw  err_st={sc_ref["err_st_mm"]:.3f}')
ax.plot(r, curves_cal['QVAL_mixmax'], color='darkorange', lw=1.8, label=f'Cal  err_st={curves_cal["err_st_mm"]:.3f}')
ax.set_title('FDR: raw pool vs calibrated'); ax.set_xlabel('Fraction accepted')
ax.set_ylabel('q-value'); ax.legend(fontsize=9); ax.grid(ls='--', alpha=0.3)

ax = axes[1]
ax.plot(r, sc_ref['Acc_true'],        'k--', lw=2,   label='True Acc')
ax.plot(r, sc_ref['Acc_est_MM'],      color='#1976D2',   lw=1.8, label=f'Raw  err_ta={sc_ref["err_ta_mm"]:.3f}')
ax.plot(r, curves_cal['Acc_est_MM'],  color='darkorange', lw=1.8, label=f'Cal  err_ta={curves_cal["err_ta_mm"]:.3f}')
ax.set_title('Acc: raw pool vs calibrated'); ax.set_xlabel('Fraction accepted')
ax.set_ylabel('Accuracy'); ax.legend(fontsize=9); ax.grid(ls='--', alpha=0.3)

plt.suptitle('Plant — Effect of pool calibration on FDR/Acc', fontsize=13)
plt.tight_layout(); plt.show()


### MAE summary — raw strategies

In [ ]:
print('='*60)
print('MAE SUMMARY — Raw Decoy Strategies')
print('='*60)
print(f"  {'Strategy':<25} {'MAE_ST':>8}  {'MAE_TA':>8}")
print(f"  {'-'*45}")
for strat_name, _ in STRATEGIES:
    c = raw_results[strat_name]
    print(f"  {STRATEGY_LABELS[strat_name]:<25} {c['err_st_mm']:>8.4f}  {c['err_ta_mm']:>8.4f}")
print(f"  {'Calibrated (SC)':<25} {curves_cal['err_st_mm']:>8.4f}  {curves_cal['err_ta_mm']:>8.4f}")


## Flow training — all 4 strategies

Trains a separate normalizing flow for each decoy strategy. Saves checkpoints to `plant_flow_{strategy}_half.pth`.

In [ ]:
FLOW_EPOCHS    = 30
FLOW_LR        = 3e-4
FLOW_PATIENCE  = 5
FLOW_N         = 12
FLOW_ENC_DIM   = 128
FLOW_SUBSAMPLE = 0.5
FLOW_SEED      = 42

n_total = len(train_scores_raw)
n_flow  = int(n_total * FLOW_SUBSAMPLE)
sub_idx = np.random.default_rng(FLOW_SEED).choice(n_total, size=n_flow, replace=False)
sub_idx.sort()

flow_tr_sc = train_scores_raw[sub_idx]
flow_tr_ft = train_features.numpy()[sub_idx]
flow_tr_lb = train_labels_raw[sub_idx]
print(f"Flow subset: {n_flow}/{n_total} ({FLOW_SUBSAMPLE*100:.0f}%)"
      f"  acc={(flow_tr_sc.argmax(1)==flow_tr_lb).mean():.4f}")

flow_results = {}
for strat_name, noise_std in STRATEGIES:
    print(f'\n{"#"*55}  {strat_name}  (noise={noise_std})')
    train_decoy = apply_strategy(flow_tr_sc, pool_score, pool_error_vectors, strat_name, noise_std)
    train_ds = ScoreFeatureDataset(
        torch.from_numpy(flow_tr_sc).float(), torch.from_numpy(flow_tr_ft).float(),
        torch.from_numpy(train_decoy).float(), torch.from_numpy(flow_tr_lb).long())

    flow_path = f'plant_flow_{strat_name}_half.pth'
    flow = ScoreShiftFlowWrapper(NUM_CLASSES, FLOW_N, FEATURE_DIM, 256, FLOW_ENC_DIM, 5.0).to(DEVICE)

    if os.path.exists(flow_path):
        flow.load_state_dict(torch.load(flow_path, map_location=DEVICE, weights_only=False))
        print(f"  Loaded from {flow_path}")
    else:
        print(f"  Training → {flow_path}")
        flow.train_flow(train_ds, epochs=FLOW_EPOCHS, lr=FLOW_LR,
                        batch_size=256, device=str(DEVICE),
                        patience=FLOW_PATIENCE, grad_clip=1.0)
        torch.save(flow.state_dict(), flow_path)
        print(f"  Saved → {flow_path}")

    flow.eval()
    tset_dc = apply_strategy(sc_np, pool_score, pool_error_vectors, strat_name, noise_std)
    test_ds = ScoreFeatureDataset(
        torch.from_numpy(sc_np).float(), torch.from_numpy(ft_np).float(),
        torch.from_numpy(tset_dc).float(), torch.from_numpy(lb_np).long())
    ms_np, ds_np, ls_np = flow.generate_decoys(test_ds, device=str(DEVICE))
    crv = compute_fdr_acc_curves(ms_np, ds_np, ls_np)
    flow_results[strat_name] = crv
    print(f"  true_acc={crv['true_acc']:.3f}  err_st={crv['err_st_mm']:.3f}  err_ta={crv['err_ta_mm']:.3f}")

print('\nAll flow experiments done.')


### Raw vs Flow — FDR and Accuracy curves

In [ ]:
r = raw_results['score_coord']['normalized_rank']
fig, axes = plt.subplots(2, n_strats, figsize=(5*n_strats, 8))
for col, (strat_name, _) in enumerate(STRATEGIES):
    rc = raw_results[strat_name]; fc = flow_results[strat_name]; color = STRATEGY_COLORS[strat_name]

    ax = axes[0][col]
    ax.plot(r, rc['QVAL_true'],    color='black',      lw=2,   ls='--', label='True FDR')
    ax.plot(r, rc['QVAL_mixmax'],  color=color,        lw=1.8,          label=f'Raw  {rc["err_st_mm"]:.3f}')
    ax.plot(r, fc['QVAL_mixmax'],  color='darkorange', lw=1.8, ls='-.', label=f'Flow {fc["err_st_mm"]:.3f}')
    ax.set_title(STRATEGY_LABELS[strat_name]); ax.set_xlabel('Fraction accepted')
    ax.legend(fontsize=7); ax.grid(ls='--', alpha=0.3); ax.set_ylim(0, 1.05)
    if col == 0: ax.set_ylabel('q-value (FDR)')

    ax = axes[1][col]
    ax.plot(r, rc['Acc_true'],     color='black',      lw=2,   ls='--', label='True Acc')
    ax.plot(r, rc['Acc_est_MM'],   color=color,        lw=1.8,          label=f'Raw  {rc["err_ta_mm"]:.3f}')
    ax.plot(r, fc['Acc_est_MM'],   color='darkorange', lw=1.8, ls='-.', label=f'Flow {fc["err_ta_mm"]:.3f}')
    ax.set_xlabel('Fraction accepted'); ax.legend(fontsize=7); ax.grid(ls='--', alpha=0.3)
    if col == 0: ax.set_ylabel('Accuracy')

plt.suptitle('Plant — Raw vs Flow decoys (all strategies)', fontsize=13)
plt.tight_layout(); plt.show()


### Raw vs Flow — score distributions

In [ ]:
ref = raw_results['score_coord']
bins = np.linspace(ref['pred_scores'].min() - 0.3, ref['pred_scores'].max() + 0.3, 60)
fig, axes = plt.subplots(2, n_strats, figsize=(5*n_strats, 8))
for col, (strat_name, _) in enumerate(STRATEGIES):
    rc = raw_results[strat_name]; fc = flow_results[strat_name]; color = STRATEGY_COLORS[strat_name]
    for row, (c, lbl, col2) in enumerate([(rc, 'RAW', color), (fc, 'FLOW', 'darkorange')]):
        ax = axes[row][col]
        sns.histplot(c['pred_scores'],  bins=bins, stat='density', color='steelblue',
                     kde=True, fill=True, alpha=0.3, label='model', ax=ax)
        sns.histplot(c['decoy_scores'], bins=bins, stat='density', color=col2,
                     kde=True, fill=True, alpha=0.4, label='decoy', ax=ax)
        ax.set_title(f'{STRATEGY_LABELS[strat_name]}  {lbl}  err_st={c["err_st_mm"]:.3f}')
        ax.set_xlabel('Max logit'); ax.legend(fontsize=7)
        if col == 0: ax.set_ylabel(f'Density ({lbl.lower()})')
plt.suptitle('Plant — Raw vs Flow score distributions', fontsize=13)
plt.tight_layout(); plt.show()


### Final MAE summary — Raw vs Flow

In [ ]:
print('='*70)
print('MAE SUMMARY — Raw vs Flow (all strategies)')
print('='*70)
print(f"  {'Strategy':<25} {'Raw ST':>8}  {'Raw TA':>8}  {'Flow ST':>8}  {'Flow TA':>8}")
print(f"  {'-'*65}")
for strat_name, _ in STRATEGIES:
    rc = raw_results[strat_name]; fc = flow_results[strat_name]
    print(f"  {STRATEGY_LABELS[strat_name]:<25} "
          f"{rc['err_st_mm']:>8.4f}  {rc['err_ta_mm']:>8.4f}  "
          f"{fc['err_st_mm']:>8.4f}  {fc['err_ta_mm']:>8.4f}")
print(f"  {'Calibrated pool (SC)':<25} {curves_cal['err_st_mm']:>8.4f}  {curves_cal['err_ta_mm']:>8.4f}")
